<a href="https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/10_decision_audit/planner_practice_method_benchmark_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Planner-Practice Method Benchmark (Colab)

## Objective
Run a single-backend (LatentDriver) benchmark across many risk/controller variants inspired by standard practices in recent papers:
- native-style risk-constrained gating (chance-constrained style)
- risk-penalized reranking (RACP/RAP-style)
- conformal effective-threshold control
- proxy and calibrated proxy variants

Heavy logic is implemented in `src/workflows/planner_method_variant_flow.py`.


## Hypotheses Tested

1. Controller family choice changes safety-progress tradeoffs even with fixed candidate sets.
2. Calibrated variants improve threshold decision correctness relative to raw variants.
3. Method-inspired risk formulations (RACP-style / chance-constrained style / reachability-style) are not equivalent in decision behavior.
4. Bottleneck attribution (signal vs rule vs candidate quality) varies across planner variants/runs.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

NOTEBOOK_NAME = 'planner_practice_method_benchmark_colab'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for p in (REPO_DIR, f'{REPO_DIR}/src'):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True. Re-run this notebook after runtime restart.')


## Step 4 - Configuration + Run Context


In [ ]:
import numpy as np

from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v2')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)

cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('run_prefix (flat artifacts) =', cfg.run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_risk_uq_run_context(run_ctx, display_fn=display)

FOCUS_LABEL = str(run_cfg.get('focus_label', 'failure_proxy_h15'))
RISK_BUDGET_TAU = float(run_cfg.get('risk_budget_tau', 0.20))
TAU_SWEEP_VALUES = [round(float(x), 3) for x in list(np.linspace(0.05, 0.80, 16))]
BOOTSTRAP_SAMPLES = int(run_cfg.get('decision_bootstrap_samples', 300))
BOOTSTRAP_SEED = int(run_cfg.get('decision_bootstrap_seed', 17))
LOCAL_TAU_WINDOW = float(run_cfg.get('local_tau_window', 0.05))

ANALYSIS_RUN_PREFIX = str(run_cfg.get('analysis_run_prefix', '')).strip()
ANALYSIS_RUN_PREFIXES = run_cfg.get('analysis_run_prefixes', [])
ALLOW_PREVIOUS_RUN_FALLBACK = bool(run_cfg.get('allow_previous_run_fallback', True))
MAX_DISCOVERED_RUNS = int(max(1, int(run_cfg.get('max_discovered_probe_runs', 50))))
MAX_RUNS_TO_LOAD = int(max(1, int(run_cfg.get('max_probe_runs_to_load', 8))))

CONTROLLER_VARIANTS = run_cfg.get(
    'controller_risk_variants',
    ['combo:platt', 'combo:raw', 'racp_style:platt', 'ccmpc_style:platt', 'radius_style:platt'],
)
RACP_LAMBDAS = run_cfg.get('racp_lambda_values', [0.0, 0.5, 1.0, 2.0, 4.0])
CONFORMAL_MIN_CAL_ROWS = int(max(30, int(run_cfg.get('conformal_min_cal_rows', 80))))
CONFORMAL_MIN_ACCEPT = int(max(10, int(run_cfg.get('conformal_min_accept_count', 20))))
CONFORMAL_SHIFT_LOCAL = bool(run_cfg.get('conformal_shift_local', False))

PAPER_REPO_ROOT = str(run_cfg.get('paper_repo_root', '/content/drive/MyDrive/waymax_experiments/paper_refs')).strip()
FETCH_PAPER_REPOS = bool(run_cfg.get('fetch_paper_repos', False))


## Step 5 - Optional: Fetch Paper Code Snapshots


In [ ]:
from pathlib import Path

fetch_script = Path(REPO_DIR) / 'experiments' / 'risk-uq-suite' / 'references' / 'fetch_inspiration_repos.sh'
if FETCH_PAPER_REPOS:
    Path(PAPER_REPO_ROOT).mkdir(parents=True, exist_ok=True)
    subprocess.run(['bash', str(fetch_script), PAPER_REPO_ROOT], check=True)
    print('[paper repos] fetched to', PAPER_REPO_ROOT)
else:
    print('[paper repos] fetch disabled; set fetch_paper_repos=true to clone snapshots')


## Step 6 - Run Planner-Practice Method Benchmark Flow


In [ ]:
from src.workflows import run_planner_method_variant_audit

audit_bundle = run_planner_method_variant_audit(
    output_prefix=str(cfg.run_prefix),
    current_run_prefix=str(cfg.run_prefix),
    persist_root=PERSIST_ROOT,
    focus_label=FOCUS_LABEL,
    risk_budget_tau=RISK_BUDGET_TAU,
    analysis_run_prefix=ANALYSIS_RUN_PREFIX,
    analysis_run_prefixes=ANALYSIS_RUN_PREFIXES,
    allow_previous_run_fallback=ALLOW_PREVIOUS_RUN_FALLBACK,
    max_discovered_runs=MAX_DISCOVERED_RUNS,
    max_runs_to_load=MAX_RUNS_TO_LOAD,
    tau_values=TAU_SWEEP_VALUES,
    local_tau_window=LOCAL_TAU_WINDOW,
    uq_probability_bins=int(max(10, int(getattr(cfg, 'uq_eval_probability_bins', 15)))),
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    bootstrap_seed=BOOTSTRAP_SEED,
    controller_variant_keys=CONTROLLER_VARIANTS,
    racp_lambda_values=RACP_LAMBDAS,
    conformal_min_cal_rows=CONFORMAL_MIN_CAL_ROWS,
    conformal_min_accept_count=CONFORMAL_MIN_ACCEPT,
    conformal_shift_local=CONFORMAL_SHIFT_LOCAL,
    reference_repo_root=PAPER_REPO_ROOT,
    write_artifacts=True,
)

print('loaded_run_prefixes =', audit_bundle.loaded_run_prefixes)
print('planner_variants =', sorted(audit_bundle.predictions_df['planner_variant'].astype(str).unique().tolist()))
print('signal_metric_rows =', len(audit_bundle.signal_metrics_df))
print('controller_metric_rows =', len(audit_bundle.controller_metrics_df))

display(audit_bundle.loaded_sources_df.head(20))
display(audit_bundle.repo_reference_df)
display(audit_bundle.signal_metrics_df.head(25))
display(audit_bundle.controller_at_budget_df.head(40))
display(audit_bundle.diagnosis_df)


## Step 7 - Quick Comparison Plots


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Plot 1: Best nominal ECE per planner.
nom = audit_bundle.signal_metrics_df[audit_bundle.signal_metrics_df['domain'].astype(str).eq('nominal')].copy()
if not nom.empty:
    best = (
        nom.sort_values(['planner_variant', 'ece', 'nll'])
        .groupby('planner_variant', as_index=False)
        .first()
    )
    fig, ax = plt.subplots(1, 1, figsize=(8, 4.8))
    ax.bar(best['planner_variant'].astype(str), best['ece'].astype(float))
    ax.set_title('Best nominal ECE per planner variant')
    ax.set_xlabel('planner_variant')
    ax.set_ylabel('ECE')
    ax.grid(axis='y', alpha=0.25)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

# Plot 2: Progress vs failure for key controllers at tau budget.
budget = audit_bundle.controller_at_budget_df.copy()
if not budget.empty:
    keep = budget[budget['controller'].astype(str).isin([
        'current', 'chance:combo:raw', 'chance:combo:platt', 'conformal:combo:platt', 'oracle-risk', 'oracle-best'
    ])].copy()
    if not keep.empty:
        fig, ax = plt.subplots(1, 1, figsize=(8, 5.2))
        for ctrl, sub in keep.groupby('controller', sort=False):
            ax.scatter(sub['progress_mean'], sub['failure_rate'], label=str(ctrl), s=40)
        ax.set_title('Controller tradeoff at tau budget')
        ax.set_xlabel('mean progress')
        ax.set_ylabel('failure rate')
        ax.grid(alpha=0.25)
        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()


## Step 8 - Contract Manifest + Mirror


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

extra = {
    'analysis_run_prefix': str(ANALYSIS_RUN_PREFIX),
    'focus_label': str(FOCUS_LABEL),
    'risk_budget_tau': float(RISK_BUDGET_TAU),
    'bootstrap_samples': int(BOOTSTRAP_SAMPLES),
    'planner_count': int(audit_bundle.predictions_df['planner_variant'].nunique()) if not audit_bundle.predictions_df.empty else 0,
    'signal_metric_rows': int(len(audit_bundle.signal_metrics_df)),
    'controller_metric_rows': int(len(audit_bundle.controller_metrics_df)),
    'artifact_count': int(len(audit_bundle.artifact_paths)),
}

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name=NOTEBOOK_NAME,
    stage='planner_practice_method_benchmark_completed',
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage='planner_practice_method_benchmark_completed',
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=dict(audit_bundle.artifact_paths),
    metrics_csv_path=audit_bundle.artifact_paths.get('planner_variant_controller_metrics', ''),
    extra_fields=extra,
)

print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
for k, v in sorted(audit_bundle.artifact_paths.items()):
    print('-', k, '->', v)


## Step 9 - Final Summary


In [ ]:
print('=== Final Summary (Planner-Practice Method Benchmark) ===')
print('focus_label =', FOCUS_LABEL, '| tau =', RISK_BUDGET_TAU)
print('loaded_runs =', len(audit_bundle.loaded_run_prefixes))

if not audit_bundle.signal_metrics_df.empty:
    best_nom = (
        audit_bundle.signal_metrics_df[audit_bundle.signal_metrics_df['domain'].astype(str).eq('nominal')]
        .sort_values(['planner_variant', 'ece', 'nll', 'brier'])
        .groupby('planner_variant', as_index=False)
        .first()
    )
    print('\n[best nominal signal variant per planner]')
    display(best_nom[['planner_variant', 'signal', 'calibration', 'auroc', 'auprc', 'ece', 'nll', 'brier']])

if not audit_bundle.controller_at_budget_df.empty:
    print('\n[controller diagnostics near tau budget]')
    display(audit_bundle.controller_at_budget_df.head(60))

if not audit_bundle.diagnosis_df.empty:
    print('\n[bottleneck diagnosis]')
    display(audit_bundle.diagnosis_df)

print('\nInterpretation guideline:')
print('- Signal rows test proxy quality and calibration quality.')
print('- Controller rows test decision-rule impact with fixed candidate sets.')
print('- Oracle rows bound what is achievable with current candidates.')
